In [ ]:
!pip install pulp

In [ ]:
import pulp

# Init the LP problem (Maximization)
prob = pulp.LpProblem("Maximize_Product_Flow", pulp.LpMaximize)

# Graph data: capacities (a_i) and demands (b_j)
supply_nodes = {'u1': 3.5, 'u2': 2.0, 'u3': 4.0, 'u4': 1.5}
demand_nodes = {'v1': 2.5, 'v2': 3.0, 'v3': 2.0, 'v4': 1.5, 'v5': 2.0}

# Valid edges according to the given graph
edges = [
    ('u1', 'v1'), ('u1', 'v2'), ('u1', 'v4'),
    ('u2', 'v2'), ('u2', 'v5'),
    ('u3', 'v3'), ('u3', 'v4'),
    ('u4', 'v2'), ('u4', 'v5')
]

# Decision variables: flow on each edge, must be >= 0
x = pulp.LpVariable.dicts("flow", edges, lowBound=0, cat='Continuous')

# Objective function: Maximize total flow
prob += pulp.lpSum([x[e] for e in edges]), "Total_Flow_Objective"

# Constraints

# Supply constraints: outgoing flow from u <= capacity of u
for u in supply_nodes:
    prob += pulp.lpSum([x[(u_i, v)] for (u_i, v) in edges if u_i == u]) <= supply_nodes[u], f"Supply_Constraint_{u}"

# Demand constraints: incoming flow to v <= demand of v
for v in demand_nodes:
    prob += pulp.lpSum([x[(u, v_i)] for (u, v_i) in edges if v_i == v]) <= demand_nodes[v], f"Demand_Constraint_{v}"

# Solve the LP
prob.solve()

# Print the results
print("Status:", pulp.LpStatus[prob.status])
print("\n Flow on edges ")
for v in prob.variables():
    # Only print edges with actual flow
    if v.varValue > 0:
        # Remove underscores from the variable name for cleaner output
        clean_name = v.name.replace("_", "")
        print(f"{clean_name} = {v.varValue}")

print("\n Maximum Total Flow ")
print(f"Total Flow = {pulp.value(prob.objective)}")

Status: Optimal

 Flow on edges 
flow('u1','v1') = 2.5
flow('u1','v2') = 1.0
flow('u2','v5') = 2.0
flow('u3','v3') = 2.0
flow('u3','v4') = 1.5
flow('u4','v2') = 1.5

 Maximum Total Flow 
Total Flow = 10.5
